## Problem Statement

### Business Context

In today's dynamic business landscape, organizations are increasingly recognizing the pivotal role customer feedback plays in shaping the trajectory of their products and services. The ability to swiftly and effectively respond to customer input not only fosters enhanced customer experiences but also serves as a catalyst for growth, prolonged customer engagement, and the nurturing of lifetime value relationships. As a dedicated Product Manager or Product Analyst, staying attuned to the voice of your customers is not just a best practice; it's a strategic imperative.

While your organization may be inundated with a wealth of customer-generated feedback and support tickets, your role entails much more than just processing these inputs. To make your efforts in managing customer experience and expectations truly impactful, you need a structured approach – a method that allows you to discern the most pressing issues, set priorities, and allocate resources judiciously. One of the most effective strategies at your disposal is to harness the power of Support Ticket Categorization.


### Objective

Develop an advanced support ticket categorization system that accurately classifies incoming tickets, assigns relevant tags based on their content, implements mechanisms and generate the first response based on the sentiment for prioritizing tickets for prompt resolution.


## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --upgrade --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 59.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 122.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 162.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 201.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lida 0.0.10 requires fastapi, which is not installed.
lida 0.0.10 requires kaleido, which is not installed.
lida 0.0.10 requires python-multipart, which is not installed.
lida 0.0.10 requires uvicorn, which is not installed.
llmx 0.0.15a0 requires cohere, which is not installed.
llmx 0.0.15a0 requires openai, which i

In [2]:
# For downloading the models from HF Hub
!pip install huggingface_hub -q

In [3]:
# Importing library for data manipulation
import pandas as pd

# Function to download the model from the Hugging Face model hub
from huggingface_hub import hf_hub_download

# Importing the Llama class from the llama_cpp module
from llama_cpp import Llama

# Importing the json module
import json

## Loading the Data

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
data = pd.read_csv("/content/support_ticket_data.csv")

## Data Overview

### Checking the first 5 rows of the data

In [ ]:
data.head()

,support_tick_id,support_ticket_text
0,ST2023-006,My internet connection has significantly slowe...
1,ST2023-007,Urgent help required! My laptop refuses to sta...
2,ST2023-008,I've accidentally deleted essential work docum...
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...
4,ST2023-010,"My smartphone battery is draining rapidly, eve..."


### Checking the shape of the data

In [ ]:
data.shape

(21, 2)

* There are 21 rows and 2 columns

### Checking the missing values in the data

In [ ]:
data.isnull().sum()

support_tick_id        0
support_ticket_text    0
dtype: int64

* There are no missing values in the data

## Model Building

### Loading the model

In [6]:
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf" # the model is in gguf format

In [7]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
lcpp_llm = Llama(
    model_path=model_path,
    n_threads=2,  # CPU cores
    n_batch=512,  # Should be between 1 and n_ctx, consider the amount of VRAM in GPU.
    n_gpu_layers=43,  # Change this value based on model and GPU VRAM pool.
    n_ctx=4096,  # Context window
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 
Model metadata: {'tokenizer.ggml.unknown_token_id': '0', 'tokenizer.ggml.eos_token_id': '2', 'general.architecture': 'llama', 'llama.context_length': '4096', 'general.name': 'LLaMA v2', 'llama.embedding_length': '5120', 'llama.feed_forward_length': '13824', 'llama.attention.layer_norm_rms_epsilon': '0.000010', 'llama.rope.dimension_count': '128', 'llama.attention.head_count': '40', 'tokenizer.ggml.bos_token_id': '1', 'llama.block_count': '40', 'llama.attention.head_count_kv': '40', 'general.quantization_version': '2', 'tokenizer.ggml.model': 'llama', 'general.file_type': '17'}


### Defining Model Response Parameters

In [9]:
def generate_llama_response(instruction, review):

    # System message explicitly instructing not to include the review text
    system_message = """
        [INST]<<SYS>>
        {}
        <</SYS>>[/INST]
    """.format(instruction)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"{review}\n{system_message}"

    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=1024,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False,
        seed=42,
    )

    # Extract the sentiment from the response
    response_text = response["choices"][0]["text"]
    return response_text

## 1. Ticket Categorization

In [ ]:
# creating a copy of the data
data_1 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_1 = """
    You are a technical assistant. Classify the support ticket based on the Support Ticket Text presented
    in the input into the following categories
      - Technical issues
      - Hardware issues
      - Data recovery
"""

In [ ]:
data_1['model_response'] = data_1['support_ticket_text'].apply(lambda x: generate_llama_response(instruction_1, x))

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [ ]:
data_1['model_response'].head()

0     Based on the provided support ticket text, I ...
1     Based on the information provided, I would cl...
2     Based on the input, I would classify this sup...
3     Based on the provided support ticket text, I ...
4     Based on the information provided, I would cl...
Name: model_response, dtype: object

In [ ]:
i = 2
print(data_1.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [ ]:
print(data_1.loc[i, 'model_response'])

 Based on the input, I would classify this support ticket as:

Data Recovery


In [ ]:
def extract_category(model_response):
    if 'technical issues' in model_response.lower():
        return 'Technical issues'
    elif 'hardware issues' in model_response.lower():
        return 'Hardware issues'
    elif 'data recovery' in model_response.lower():
        return 'Data recovery'

In [ ]:
# applying the function to the model response
data_1['Category'] = data_1['model_response'].apply(extract_category)
data_1['Category'].head()

0               None
1    Hardware issues
2      Data recovery
3      Data recovery
4    Hardware issues
Name: Category, dtype: object

In [ ]:
data_1['Category'].value_counts()

Data recovery       8
Hardware issues     6
Technical issues    4
Name: Category, dtype: int64

In [ ]:
final_data_1 = data_1.drop(['model_response'], axis=1)
final_data_1.head()

,support_tick_id,support_ticket_text,Category
0,ST2023-006,My internet connection has significantly slowe...,None
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware issues
2,ST2023-008,I've accidentally deleted essential work docum...,Data recovery
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Data recovery
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware issues


* Only one support ticket in the data did not receive any category label.

## 2: Ticket Categorization and Returning Structured Output

In [10]:
# creating a copy of the data
data_2 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_2 = """
    You are a technical assistant. Classify the support ticket based on the Support Ticket Text presented
    in the input into the following categories
      - Technical issues
      - Hardware issues
      - Data recovery

    Format the output as a JSON object with a single key-value pair as shown below:
    {"Category": "your_category_prediction"}
"""

In [ ]:
data_2['model_response'] = data_2['support_ticket_text'].apply(lambda x: generate_llama_response(instruction_2, x))

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [ ]:
data_2['model_response'].head()

0     Sure, I'd be happy to help! Based on the inpu...
1                  {\n"Category": "Hardware Issues"\n}
2     Sure, I'd be happy to help! Based on the inpu...
3     Sure, I'd be happy to help! Based on the inpu...
4     Sure, I'd be happy to help! Based on the inpu...
Name: model_response, dtype: object

In [ ]:
i = 2
print(data_2.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [ ]:
print(data_2.loc[i, 'model_response'])

 Sure, I'd be happy to help! Based on the input you provided, I would classify this support ticket as follows:

{"Category": "Data recovery"}


In [10]:
# defining a function to parse the JSON output from the model
def extract_json_data(json_str):
    try:
        # Find the indices of the opening and closing curly braces
        json_start = json_str.find('{')
        json_end = json_str.rfind('}')

        if json_start != -1 and json_end != -1:
            extracted_category = json_str[json_start:json_end + 1]  # Extract the JSON object
            data_dict = json.loads(extracted_category)
            return data_dict
        else:
            print(f"Warning: JSON object not found in response: {json_str}")
            return {}
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return {}

In [ ]:
# Applying the function to the model response
data_2['model_response_parsed'] = data_2['model_response'].apply(extract_json_data)
data_2['model_response_parsed'].head()

0    {'Category': 'Technical issues'}
1     {'Category': 'Hardware Issues'}
2       {'Category': 'Data recovery'}
3    {'Category': 'Technical issues'}
4     {'Category': 'Hardware issues'}
Name: model_response_parsed, dtype: object

In [ ]:
# Normalizing the model_response_parsed column
model_response_parsed_df_2 = pd.json_normalize(data_2['model_response_parsed'])
model_response_parsed_df_2.head()

,Category
0,Technical issues
1,Hardware Issues
2,Data recovery
3,Technical issues
4,Hardware issues


In [ ]:
# Concatinating two dataframes
data_with_parsed_model_output_2 = pd.concat([data_2, model_response_parsed_df_2], axis=1)
data_with_parsed_model_output_2.head()

,support_tick_id,support_ticket_text,model_response,model_response_parsed,Category
0,ST2023-006,My internet connection has significantly slowe...,"Sure, I'd be happy to help! Based on the inpu...",{'Category': 'Technical issues'},Technical issues
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"{\n""Category"": ""Hardware Issues""\n}",{'Category': 'Hardware Issues'},Hardware Issues
2,ST2023-008,I've accidentally deleted essential work docum...,"Sure, I'd be happy to help! Based on the inpu...",{'Category': 'Data recovery'},Data recovery
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"Sure, I'd be happy to help! Based on the inpu...",{'Category': 'Technical issues'},Technical issues
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","Sure, I'd be happy to help! Based on the inpu...",{'Category': 'Hardware issues'},Hardware issues


In [ ]:
# Dropping model_response and model_response_parsed columns
final_data_2 = data_with_parsed_model_output_2.drop(['model_response','model_response_parsed'], axis=1)
final_data_2.head()

,support_tick_id,support_ticket_text,Category
0,ST2023-006,My internet connection has significantly slowe...,Technical issues
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware Issues
2,ST2023-008,I've accidentally deleted essential work docum...,Data recovery
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Technical issues
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware issues


In [ ]:
# Checking the value counts of Category column
final_data_2['Category'].value_counts()

Technical issues    7
Data recovery       4
Hardware issues     4
Data Recovery       3
Hardware Issues     2
Hardware            1
Name: Category, dtype: int64

* We can observe that there are more than 3 categories with different spellings

## 3: Ticket Categorization, Creating Tags, and Returning Structured Output

In [ ]:
# creating a copy of the data
data_3 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_3 = """
    You are a technical assistant. Classify the support ticket based on the Support Ticket Text presented
    in the input into the following categories.
      - Technical issues
      - Hardware issues
      - Data recovery

    Once you have identified the Category, then :
        1) Create Tags - This will help us classify the ticket and don't include the category

        Output the category and tags for each category in a JSON format with the following keys:
    {
        "Category": "categories",
        "Tags": "Tag"
    }
"""

In [ ]:
# Applying generate_llama_response function on support_ticket_text column
data_3['model_response'] = data_3['support_ticket_text'].apply(lambda x: generate_llama_response(instruction_3, x))

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [ ]:
data_3['model_response'].head()

0     Sure, I can help you classify the support tic...
1     Sure, I can help you classify your support ti...
2     Sure, I can help you with that! Based on the ...
3     Sure, I can help you classify your support ti...
4     Sure, I can help you identify and rectify you...
Name: model_response, dtype: object

In [ ]:
i = 2
print(data_3.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [ ]:
print(data_3.loc[i, 'model_response'])

 Sure, I can help you with that! Based on the input provided, I would classify this support ticket as a Data Recovery issue. Here are the tags for each category in JSON format:

{
"Category": "Data Recovery",
"Tags": ["data loss", "essential documents", "substantial data loss"]
}


In [ ]:
# Applying the function to the model response
data_3['model_response_parsed'] = data_3['model_response'].apply(extract_json_data)
data_3['model_response_parsed'].head()


Category: Data Recovery

Tags: #DataRecovery #USBFlashDrive #PhysicalDamage #FileRecovery


0    {'Category': 'Technical Issues', 'Tags': ['Int...
1    {'Category': 'Hardware Issues', 'Tags': 'Lapto...
2    {'Category': 'Data Recovery', 'Tags': ['data l...
3    {'Category': 'Technical Issues', 'Tags': ['Wi-...
4    {'Category': 'Hardware Issues', 'Tags': ['Batt...
Name: model_response_parsed, dtype: object

In [ ]:
# Normalizing the model_response_parsed column
model_response_parsed_df_3 = pd.json_normalize(data_3['model_response_parsed'])
model_response_parsed_df_3.head(21)

,Category,Tags,Category.Technical Issues,Category.Hardware Issues,Category.Data Recovery,Tags.Urgent,Tags.Password Reset
0,Technical Issues,"[Internet Connection, Speed, Disconnections]",NaN,NaN,NaN,NaN,NaN
1,Hardware Issues,"Laptop, Startup, Restart, Hardware Failure",NaN,NaN,NaN,NaN,NaN
2,Data Recovery,"[data loss, essential documents, substantial d...",NaN,NaN,NaN,NaN,NaN
3,Technical Issues,"[Wi-Fi Signal Strength, Router Positioning, In...",NaN,NaN,NaN,NaN,NaN
4,Hardware Issues,"[Battery Life, Smartphone Battery]",NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,[I'm locked out of my online banking account a...,[],[],[urgent transaction],[password reset]
6,Technical,"[Performance, Optimization]",NaN,NaN,NaN,NaN,NaN
7,Hardware,"[blue screen error, frequent crashes]",NaN,NaN,NaN,NaN,NaN
8,Data Recovery,"[Hardware Issues, Technical Issues]",NaN,NaN,NaN,NaN,NaN
9,Hardware Issues,"[Graphics Card, Malfunctioning, Poor Gaming Pe...",NaN,NaN,NaN,NaN,NaN


In [ ]:
model_response_parsed_df_3.drop(['Category.Technical Issues','Category.Hardware Issues','Category.Data Recovery','Tags.Urgent','Tags.Password Reset'], axis=1,inplace=True)

In [ ]:
# Concatinating two dataframes
data_with_parsed_model_output_3 = pd.concat([data_3, model_response_parsed_df_3], axis=1)
data_with_parsed_model_output_3.head()

,support_tick_id,support_ticket_text,model_response,model_response_parsed,Category,Tags
0,ST2023-006,My internet connection has significantly slowe...,"Sure, I can help you classify the support tic...","{'Category': 'Technical Issues', 'Tags': ['Int...",Technical Issues,"[Internet Connection, Speed, Disconnections]"
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"Sure, I can help you classify your support ti...","{'Category': 'Hardware Issues', 'Tags': 'Lapto...",Hardware Issues,"Laptop, Startup, Restart, Hardware Failure"
2,ST2023-008,I've accidentally deleted essential work docum...,"Sure, I can help you with that! Based on the ...","{'Category': 'Data Recovery', 'Tags': ['data l...",Data Recovery,"[data loss, essential documents, substantial d..."
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"Sure, I can help you classify your support ti...","{'Category': 'Technical Issues', 'Tags': ['Wi-...",Technical Issues,"[Wi-Fi Signal Strength, Router Positioning, In..."
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","Sure, I can help you identify and rectify you...","{'Category': 'Hardware Issues', 'Tags': ['Batt...",Hardware Issues,"[Battery Life, Smartphone Battery]"


In [ ]:
# Dropping model_response and model_response_parsed columns
final_data_3 = data_with_parsed_model_output_3.drop(['model_response','model_response_parsed'], axis=1)
final_data_3.head()

,support_tick_id,support_ticket_text,Category,Tags
0,ST2023-006,My internet connection has significantly slowe...,Technical Issues,"[Internet Connection, Speed, Disconnections]"
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware Issues,"Laptop, Startup, Restart, Hardware Failure"
2,ST2023-008,I've accidentally deleted essential work docum...,Data Recovery,"[data loss, essential documents, substantial d..."
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Technical Issues,"[Wi-Fi Signal Strength, Router Positioning, In..."
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware Issues,"[Battery Life, Smartphone Battery]"


## 4 - Ticket Categorization, Creating Tags, Assigning Priority, and Returning Structured Output

In [ ]:
# creating a copy of the data
data_4 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_4 = """
    You are a technical assistant. Classify the support ticket based on the Support Ticket Text presented
    in the input into the following categories
      - Technical issues
      - Hardware issues
      - Data recovery

    Once you have identified the Category, then :
        1) Create Tags - This will help us classify the ticket and don't include the category
        2) Assign Priority - The priortiy should be "High" or "Normal" based on the understanding of text

        Output the category and tags for each category in a JSON format with the following keys:
    {
        "Category": "categories",
        "Tags": "Tag",
        "Priority": "priority"
    }
"""

In [ ]:
# Applying generate_llama_response function on support_ticket_text column
data_4['model_response'] = data_4['support_ticket_text'].apply(lambda x: generate_llama_response(instruction_4, x))

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [ ]:
data_4['model_response'].head()

0     Sure, I'd be happy to help! Based on the supp...
1     Sure, I'd be happy to help! Based on the inpu...
2     Sure, I'd be happy to help! Based on the inpu...
3     Sure, I'd be happy to help! Based on the inpu...
4     Sure, I'd be happy to help! Based on the inpu...
Name: model_response, dtype: object

In [ ]:
i = 2
print(data_4.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [ ]:
print(data_4.loc[i, 'model_response'])

 Sure, I'd be happy to help! Based on the input you provided, here is the classification of your support ticket:

{
"Category": "Data Recovery",
"Tags": ["data loss", "essential documents"],
"Priority": "High"
}

Based on the information provided in your message, it appears that you have accidentally deleted important work documents and are seeking assistance with data recovery. This falls under the category of Data Recovery, so I will prioritize this ticket as High priority. The tags I have assigned include "data loss" and "essential documents" to help classify the ticket and ensure it is addressed promptly.


In [ ]:
# Applying the function to the model response
data_4['model_response_parsed'] = data_4['model_response'].apply(extract_json_data)
data_4['model_response_parsed'].head()


Category: Technical issues

Tags: internet-connection, slow-speed, disconnections

Priority: High

Here's why I classified this ticket as a technical issue with high priority:

The customer is experiencing critical problems with their internet connection, including slow speeds and frequent disconnections. This indicates a likely issue with the customer's internet service provider (ISP) or their network configuration, which requires urgent attention to minimize productivity loss and frustration. The fact that restarting the router has not resolved the problem suggests that the issue may be more complex and require specialized technical support. Therefore, I have assigned a high priority to this ticket to ensure prompt resolution.


0    {'Category': 'Technical issues', 'Tags': 'inte...
1    {'Category': 'Hardware Issues', 'Tags': 'lapto...
2    {'Category': 'Data Recovery', 'Tags': ['data l...
3    {'Category': 'Technical', 'Tags': ['Wi-Fi', 's...
4    {'Category': 'Hardware Issues', 'Tags': 'batte...
Name: model_response_parsed, dtype: object

In [ ]:
# Normalizing the model_response_parsed column
model_response_parsed_df_4 = pd.json_normalize(data_4['model_response_parsed'])
model_response_parsed_df_4.head(21)

,Category,Tags,Priority
0,Technical issues,"internet connection slow, frequent disconnections",High
1,Hardware Issues,"laptop, startup, presentation",High
2,Data Recovery,"[data loss, essential documents]",High
3,Technical,"[Wi-Fi, signal strength]",High
4,Hardware Issues,"battery issue, rapid drain",High
5,Technical issues,"[password reset, online banking]",High
6,Technical issues,"[performance, optimization]",High
7,Hardware Issues,"Blue Screen Error, Frequent Crashes",High
8,Data Recovery,"[External Hard Drive, Not Recognized, Vital Data]",High
9,Hardware Issues,"Graphics Card, Gaming Laptop, Malfunctioning",High


In [ ]:
# Concatinating two dataframes
data_with_parsed_model_output_4 = pd.concat([data_4, model_response_parsed_df_4], axis=1)
data_with_parsed_model_output_4.head()

,support_tick_id,support_ticket_text,model_response,model_response_parsed,Category,Tags,Priority
0,ST2023-006,My internet connection has significantly slowe...,"Sure, I'd be happy to help! Based on the supp...","{'Category': 'Technical issues', 'Tags': 'inte...",Technical issues,"internet connection slow, frequent disconnections",High
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"Sure, I'd be happy to help! Based on the inpu...","{'Category': 'Hardware Issues', 'Tags': 'lapto...",Hardware Issues,"laptop, startup, presentation",High
2,ST2023-008,I've accidentally deleted essential work docum...,"Sure, I'd be happy to help! Based on the inpu...","{'Category': 'Data Recovery', 'Tags': ['data l...",Data Recovery,"[data loss, essential documents]",High
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"Sure, I'd be happy to help! Based on the inpu...","{'Category': 'Technical', 'Tags': ['Wi-Fi', 's...",Technical,"[Wi-Fi, signal strength]",High
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","Sure, I'd be happy to help! Based on the inpu...","{'Category': 'Hardware Issues', 'Tags': 'batte...",Hardware Issues,"battery issue, rapid drain",High


In [ ]:
# Dropping model_response and model_response_parsed columns
final_data_4 = data_with_parsed_model_output_4.drop(['model_response','model_response_parsed'], axis=1)
final_data_4.head()

,support_tick_id,support_ticket_text,Category,Tags,Priority
0,ST2023-006,My internet connection has significantly slowe...,Technical issues,"internet connection slow, frequent disconnections",High
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware Issues,"laptop, startup, presentation",High
2,ST2023-008,I've accidentally deleted essential work docum...,Data Recovery,"[data loss, essential documents]",High
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Technical,"[Wi-Fi, signal strength]",High
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware Issues,"battery issue, rapid drain",High


## 5 - Ticket Categorization, Creating Tags, Assigning Priority, Assigning ETA, and Returning Structured Output

In [ ]:
# creating a copy of the data
data_5 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_5 = """
    You are a technical assistant. Classify the support ticket based on the Support Ticket Text presented
    in the input into the following categories
      - Technical issues
      - Hardware issues
      - Data recovery

    Once you have identified the Category, then :
        1) Create Tags - This will help us classify the ticket and don't include the category
        2) Assign Priority - The priortiy should be "High" or "Normal" based on the understanding of text
        3) Suggest ETA for resolution of the issue stated in the ticket text

        Output the category and tags for each category in a JSON format with the following keys:
    {
        "Category": "categories",
        "Tags": "Tag",
        "Priority": "priority",
        "ETA": "ETA"
    }

    Instructions:
        1. Categorise only as 'Technical issues','Hardware issues' or 'Data recovery'
"""

In [ ]:
# Applying generate_llama_response function on support_ticket_text ticket
data_5['model_response'] = data_5['support_ticket_text'].apply(lambda x: generate_llama_response(instruction_5, x))

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [ ]:
data_5['model_response'].head()

0     Sure, I can help you with that! Based on the ...
1     {\n       "Category": "Hardware Issues",\n   ...
2    \n{\n"Category": "Data Recovery",\n"Tags": "da...
3     {\n       "Category": "Technical Issues",\n  ...
4    \n    {\n        "Category": "Technical issues...
Name: model_response, dtype: object

In [ ]:
i = 2
print(data_5.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [ ]:
print(data_5.loc[i, 'model_response'])


{
"Category": "Data Recovery",
"Tags": "data loss, essential documents, prioritize",
"Priority": "High",
"ETA": "2-3 business days"
}


In [ ]:
# Applying the function to the model response
data_5['model_response_parsed'] = data_5['model_response'].apply(extract_json_data)
data_5['model_response_parsed'].head()

Error parsing JSON: Expecting ':' delimiter: line 3 column 53 (char 86)


0                                                   {}
1    {'Category': 'Hardware Issues', 'Tags': 'Lapto...
2    {'Category': 'Data Recovery', 'Tags': 'data lo...
3    {'Category': 'Technical Issues', 'Tags': 'Wi-F...
4    {'Category': 'Technical issues', 'Tags': 'batt...
Name: model_response_parsed, dtype: object

In [ ]:
# Normalizing the model_response_parsed column
model_response_parsed_df_5 = pd.json_normalize(data_5['model_response_parsed'])
model_response_parsed_df_5.head(21)

,Category,Tags,Priority,ETA
0,NaN,NaN,NaN,NaN
1,Hardware Issues,"Laptop, Start-up, Restart",High,24 hours
2,Data Recovery,"data loss, essential documents, prioritize",High,2-3 business days
3,Technical Issues,"Wi-Fi, signal strength, troubleshooting",High,24 hours
4,Technical issues,"battery issue, rapid drain",High,2-3 business days
5,Technical issues,"password reset, online banking",High,2 hours
6,Technical Issues,"performance, optimization",High,2-3 business days
7,Hardware Issues,"Blue Screen Error, PC Crashes, Hardware Diagnosis",High,2-3 business days
8,Data Recovery,"External Hard Drive, Data Recovery",High,2-3 business days
9,Hardware Issues,"Gaming Laptop, Graphics Card, Malfunctioning",High,2-3 business days


In [ ]:
# Concatinating two dataframes
data_with_parsed_model_output_5 = pd.concat([data_5, model_response_parsed_df_5], axis=1)
data_with_parsed_model_output_5.head()

,support_tick_id,support_ticket_text,model_response,model_response_parsed,Category,Tags,Priority,ETA
0,ST2023-006,My internet connection has significantly slowe...,"Sure, I can help you with that! Based on the ...",{},NaN,NaN,NaN,NaN
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"{\n ""Category"": ""Hardware Issues"",\n ...","{'Category': 'Hardware Issues', 'Tags': 'Lapto...",Hardware Issues,"Laptop, Start-up, Restart",High,24 hours
2,ST2023-008,I've accidentally deleted essential work docum...,"\n{\n""Category"": ""Data Recovery"",\n""Tags"": ""da...","{'Category': 'Data Recovery', 'Tags': 'data lo...",Data Recovery,"data loss, essential documents, prioritize",High,2-3 business days
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"{\n ""Category"": ""Technical Issues"",\n ...","{'Category': 'Technical Issues', 'Tags': 'Wi-F...",Technical Issues,"Wi-Fi, signal strength, troubleshooting",High,24 hours
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","\n {\n ""Category"": ""Technical issues...","{'Category': 'Technical issues', 'Tags': 'batt...",Technical issues,"battery issue, rapid drain",High,2-3 business days


In [ ]:
# Dropping model_response and model_response_parsed columns
final_data_5 = data_with_parsed_model_output_5.drop(['model_response','model_response_parsed'], axis=1)
final_data_5.head()

,support_tick_id,support_ticket_text,Category,Tags,Priority,ETA
0,ST2023-006,My internet connection has significantly slowe...,NaN,NaN,NaN,NaN
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware Issues,"Laptop, Start-up, Restart",High,24 hours
2,ST2023-008,I've accidentally deleted essential work docum...,Data Recovery,"data loss, essential documents, prioritize",High,2-3 business days
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Technical Issues,"Wi-Fi, signal strength, troubleshooting",High,24 hours
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Technical issues,"battery issue, rapid drain",High,2-3 business days


## 6 - Ticket Categorization, Creating Tags, Assigning Priority, Assigning ETA, Creating a Draft Response, and Returning Structured Output

In [32]:
# creating a copy of the data
data_6 = data.copy()

In [33]:
# defining the instructions for the model
instruction_6 = """
    You are a technical assistant. Classify the support ticket based on the Support Ticket Text presented
    in the input into the following categories
      - Technical issues
      - Hardware issues
      - Data recovery

    Once you have identified the Category, then :
        1) Create Tags to help us classify the ticket
            - Don't include the category name in the tags
        2) Assign Priority based on the understanding of the issue raised in the support ticket by the user
            - The priority should be "High" or "Normal"
        3) Suggest an ETA for resolution of the issue stated in the support ticket text depending upon the category and the priority
        4) Depending upon the category, priority, and ETA, draft a response that can be shared with the customer
            - When drafting the response for the customer, be polite and empathetic.
        5) Categorise tickets only as 'Technical Issues',' Hardware Issues', or 'Data Recovery'

    Return the output in the specified JSON format shared below, ensuring consistency and handling missing values appropriately. Ensure that all values in the JSON are formatted as strings, and each element within the lists should be enclosed in double quotes:
    {
        "Category": "Categories",
        "Tags": "Tag",
        "Priority": "priority",
        "ETA": "ETA",
        "Response": "response"
    }


   Only return the JSON in the output, do NOT return any other text or information.

"""

In [34]:
data_6['model_response'] = data_6['support_ticket_text'].apply(lambda x: generate_llama_response(instruction_6, x))

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


In [35]:
# Checking the head of the column
data_6['model_response'].head()

0    \n    {\n        "Category": "Technical Issues...
1    \n{\n    "Category": "Hardware Issues",\n    "...
2    \n    {\n        "Category": "Data Recovery",\...
3    \n    {\n        "Category": "Technical Issues...
4    \n    Sure! I'd be happy to help you identify ...
Name: model_response, dtype: object

In [36]:
i = 2
print(data_6.loc[i, 'support_ticket_text'])

I've accidentally deleted essential work documents, causing substantial data loss. I understand the need to avoid further actions on my device. Can you please prioritize the data recovery process and guide me through it?


In [37]:
print(data_6.loc[i, 'model_response'])


    {
        "Category": "Data Recovery",
        "Tags": ["deleted documents", "data loss"],
        "Priority": "High",
        "ETA": "2-3 business days",
        "Response": "We understand the urgency of recovering your essential work documents. Our data recovery experts will start working on your case immediately, and we expect to complete the process within 2-3 business days. Please note that data recovery is a complex process, and there are no guarantees of success. However, our team will do their best to retrieve your lost files."
    }


In [38]:
# Applying the function to the model response
data_6['model_response_parsed'] = data_6['model_response'].apply(extract_json_data)
data_6['model_response_parsed'].head()

    Sure! I'd be happy to help you identify and rectify your smartphone battery issue. Based on the support ticket text provided, here is my assessment:

Category: Technical Issues

Tags: Battery Drain, Smartphone

Priority: High

ETA: 2-3 business days (depending on the availability of replacement parts and technical expertise)

Response:
Dear [Customer], thank you for reaching out to us regarding your smartphone's rapidly draining battery. Based on the information provided, it appears that there might be a hardware issue with your device's battery. Our team will need some time to diagnose and resolve this issue, so we estimate that it may take 2-3 business days to complete the repair. In the meantime, please ensure that you have backed up any important data on your smartphone as we will need to perform a factory reset to resolve the issue. We apologize for any inconvenience caused and appreciate your patience in this matter.

Please let me know if there is anything else I can assist 

0    {'Category': 'Technical Issues', 'Tags': ['Int...
1    {'Category': 'Hardware Issues', 'Tags': ['Lapt...
2    {'Category': 'Data Recovery', 'Tags': ['delete...
3    {'Category': 'Technical Issues', 'Tags': ['Wi-...
4                                                   {}
Name: model_response_parsed, dtype: object

In [42]:
data_6[data_6.model_response_parsed == {}]

,support_tick_id,support_ticket_text,model_response,model_response_parsed
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",\n Sure! I'd be happy to help you identify ...,{}
15,ST2023-021,"My internet connection is frequently dropping,...","\n {\n ""Category"": ""Technical Issues...",{}


* There are two support tickets where none of the tasks have been extracted from the JSON function. Therefore, we will update the values manually.

In [40]:
i =4
print(data_6.loc[i, 'model_response'])


    Sure! I'd be happy to help you identify and rectify your smartphone battery issue. Based on the support ticket text provided, here is my assessment:

Category: Technical Issues

Tags: Battery Drain, Smartphone

Priority: High

ETA: 2-3 business days (depending on the availability of replacement parts and technical expertise)

Response:
Dear [Customer], thank you for reaching out to us regarding your smartphone's rapidly draining battery. Based on the information provided, it appears that there might be a hardware issue with your device's battery. Our team will need some time to diagnose and resolve this issue, so we estimate that it may take 2-3 business days to complete the repair. In the meantime, please ensure that you have backed up any important data on your smartphone as we will need to perform a factory reset to resolve the issue. We apologize for any inconvenience caused and appreciate your patience in this matter.

Please let me know if there is anything else I can assist

In [41]:
i = 15
print(data_6.loc[i, 'model_response'])


    {
        "Category": "Technical Issues",
        "Tags": ["Internet Connection"],["Droping"]],
        "Priority": "High",
        "ETA": "24 hours",
        "Response": "We understand the urgency of your issue and will escalate it to our technical team for immediate attention. We'll keep you updated on the progress and aim to resolve the matter within 24 hours."
    }


In [43]:
upd_val_1 = {
        "Category": "Technical Issues",
        "Tags": ["Battery Drain, Smartphone"],
        "Priority": "High",
        "ETA": "2-3 business days",
        "Response": "Dear [Customer], thank you for reaching out to us regarding your smartphone's rapidly draining battery. Based on the information provided, it appears that there might be a hardware issue with your device's battery. Our team will need some time to diagnose and resolve this issue, so we estimate that it may take 2-3 business days to complete the repair. In the meantime, please ensure that you have backed up any important data on your smartphone as we will need to perform a factory reset to resolve the issue. We apologize for any inconvenience caused and appreciate your patience in this matter."
    }

upd_val_2 = {
        "Category": "Technical Issues",
        "Tags": ["Internet Connection","Droping"],
        "Priority": "High",
        "ETA": "24 hours",
        "Response": "We understand the urgency of your issue and will escalate it to our technical team for immediate attention. We'll keep you updated on the progress and aim to resolve the matter within 24 hours."
    }

# defining the list of indices to update
idx_list = [4,15]
data_6.loc[idx_list, 'model_response_parsed'] = [upd_val_1, upd_val_2]

In [44]:
# Applying the function to the model response
model_response_parsed_df_6 = pd.json_normalize(data_6['model_response_parsed'])
model_response_parsed_df_6.head(21)

,Category,Tags,Priority,ETA,Response
0,Technical Issues,[Internet Connection],High,24 hours,We apologize for the inconvenience you're expe...
1,Hardware Issues,[Laptop Won't Start],High,24 hours,"I understand that your laptop is not starting,..."
2,Data Recovery,"[deleted documents, data loss]",High,2-3 business days,We understand the urgency of recovering your e...
3,Technical Issues,"[Wi-Fi, Signal Strength]",High,24 hours,We understand that you are experiencing weak W...
4,Technical Issues,"[Battery Drain, Smartphone]",High,2-3 business days,"Dear [Customer], thank you for reaching out to..."
5,Technical Issues,[password reset],High,Immediate assistance required,"Dear customer, we apologize for the inconvenie..."
6,Technical Issues,"[sluggish performance, computer optimization]",High,2-3 business days,We understand that your computer's performance...
7,Hardware Issues,"[Blue Screen Error, Frequent Crashes]",High,2-3 business days,We understand that you are experiencing freque...
8,Data Recovery,"[data recovery, external hard drive]",High,2-3 business days,We understand the urgency of recovering your v...
9,Hardware Issues,"[Graphics Card, Gaming Laptop]",High,2-3 business days,We understand that your graphics card is malfu...


In [45]:
# Concatinating two dataframes
data_with_parsed_model_output_6 = pd.concat([data_6, model_response_parsed_df_6], axis=1)
data_with_parsed_model_output_6.head()

,support_tick_id,support_ticket_text,model_response,model_response_parsed,Category,Tags,Priority,ETA,Response
0,ST2023-006,My internet connection has significantly slowe...,"\n {\n ""Category"": ""Technical Issues...","{'Category': 'Technical Issues', 'Tags': ['Int...",Technical Issues,[Internet Connection],High,24 hours,We apologize for the inconvenience you're expe...
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"\n{\n ""Category"": ""Hardware Issues"",\n ""...","{'Category': 'Hardware Issues', 'Tags': ['Lapt...",Hardware Issues,[Laptop Won't Start],High,24 hours,"I understand that your laptop is not starting,..."
2,ST2023-008,I've accidentally deleted essential work docum...,"\n {\n ""Category"": ""Data Recovery"",\...","{'Category': 'Data Recovery', 'Tags': ['delete...",Data Recovery,"[deleted documents, data loss]",High,2-3 business days,We understand the urgency of recovering your e...
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"\n {\n ""Category"": ""Technical Issues...","{'Category': 'Technical Issues', 'Tags': ['Wi-...",Technical Issues,"[Wi-Fi, Signal Strength]",High,24 hours,We understand that you are experiencing weak W...
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",\n Sure! I'd be happy to help you identify ...,"{'Category': 'Technical Issues', 'Tags': ['Bat...",Technical Issues,"[Battery Drain, Smartphone]",High,2-3 business days,"Dear [Customer], thank you for reaching out to..."


In [46]:
# Dropping model_response and model_response_parsed columns
final_data_6 = data_with_parsed_model_output_6.drop(['model_response','model_response_parsed'], axis=1)
final_data_6.head()

,support_tick_id,support_ticket_text,Category,Tags,Priority,ETA,Response
0,ST2023-006,My internet connection has significantly slowe...,Technical Issues,[Internet Connection],High,24 hours,We apologize for the inconvenience you're expe...
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware Issues,[Laptop Won't Start],High,24 hours,"I understand that your laptop is not starting,..."
2,ST2023-008,I've accidentally deleted essential work docum...,Data Recovery,"[deleted documents, data loss]",High,2-3 business days,We understand the urgency of recovering your e...
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Technical Issues,"[Wi-Fi, Signal Strength]",High,24 hours,We understand that you are experiencing weak W...
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Technical Issues,"[Battery Drain, Smartphone]",High,2-3 business days,"Dear [Customer], thank you for reaching out to..."


## Model Output Analysis

In [47]:
# Creating a copy of the dataframe of task-6
final_data = final_data_6.copy()

In [48]:
final_data.Category.value_counts()

Technical Issues    9
Data Recovery       7
Hardware Issues     5
Name: Category, dtype: int64

* The most common support category is "Technical Issues" with 9 occurrences, followed by "Data Recovery" with 7 occurrences, and "Hardware Issues" is the third most common category with 5 occurrences.

In [49]:
final_data.Priority.value_counts()

High    21
Name: Priority, dtype: int64

* The model identified all tickets to be of high priority
* One can manually read the ticket text and check if all of them are indeed high-priority

In [50]:
final_data.ETA.value_counts()

2-3 business days                15
24 hours                          5
Immediate assistance required     1
Name: ETA, dtype: int64

* "2-3 business days" is the most common ETA, with a total of 15 tickets falling into this category.
"24 hours" is the second most common ETA, with 5 tickets.

* The presence of an "Immediate assistance required" ETA for 1 ticket suggests the existence of critical issues that require immediate attention.

Let's dive in a bit deeper here.

In [51]:
final_data.groupby(['Category', 'ETA']).support_tick_id.count()

Category          ETA                          
Data Recovery     2-3 business days                7
Hardware Issues   2-3 business days                4
                  24 hours                         1
Technical Issues  2-3 business days                4
                  24 hours                         4
                  Immediate assistance required    1
Name: support_tick_id, dtype: int64

- Data Recovery:

   - All the tickets in the "Data Recovery" category have an ETA of "2-3 business days".

- Hardware Issues:

   - The majority of tickets in the "Hardware Issues" category have an ETA of "2-3 business days" (4 tickets).
   - There are also tickets with ETAs of "24 hours" (1 ticket)

- Technical Issues:

   - Tickets in the "Technical Issues" category have varied ETAs.
   - The most common ETA is "2-3 business days" (4 tickets) followed by 24 hours (4 tickets)and "Immediate assistance required" (1 ticket).

- ETA Distribution:

   - "2-3 business days" is a common ETA across different categories.
   - There's a mix of shorter and longer ETAs, indicating varying levels of urgency for issue resolution.

- Priority

    - For support tickets with ETA of 24 hours and 2-3 business days, the model has assigned a high priority, which is unlikely in the real world.